# Sequential Counter Encoding

## Giới thiệu
Cardinality Constraint (Ràng buộc số lượng) là ràng buộc kiểu "nhiều nhất, ít nhất, đúng k biến đúng".  Cách cơ bản là cấm tất cả các tổ hợp k+1 biến đúng nhưng sẽ bùng nổ tập mệnh đề khiến cho bài toán gần như không thể tìm ra nghiệm trong thời gian được giới hạn. 

Ý tưởng chung để encode một cách hiệu quả là đếm rồi so sánh. Có hai stage cơ bản:
1. Counter stage
2. Comparator stage

### Counter stage 
Counter stage là phần đếm xem có bao nhiêu biến là true (tương tự như một mảng tổng tiền tố)
Công thức như sau: $$S_i = \sum_{j=1}^{i}x_j, \quad \forall 1 \le i < n  $$

### Comparator stage
Comparator stage là phần so sánh tổng vừa đếm được với k. Nếu constraint là: $ \leq k(x_1, \dots, x_n) $ thì comparator kiểm tra: $$S_i \leq k, \quad \forall 1 \leq i < n$$


## Các định nghĩa cơ bản
Có 5 loại constraint được các tác giả định nghĩa:
1. $ \le k(\phi_{1}, \dots, \phi_{n})$: nhiều nhất $k$ công thức đúng. Ràng buộc này luông đúng với $k \ge n$ và luôn sai với $k<0$
2. $ \ge k(\phi_{1}, \dots, \phi_{n})$: ít nhất $k$ công thức đúng. Ràng buộc này luôn đúng với $k \le 0$ và luôn sai với $k>n$
3. $ = k(\phi_{1}, \dots, \phi_{n})$: đúng $k$ công thức đúng.
4. $ \le k(\phi_{1}, \dots, \phi_{n})$ tương đương $\ge (n-k)(\neg x_1, \dots, \neg x_n)$
5. $ \neg  \le k(\phi_{1}, \dots, \phi_{n})$ tương đương $\ge (k+1)(x_1,\dots,x_n)$

Có hai điều quan trọng là các tác giả đã chứng minh rằng chỉ cần tập trung vào constraint 1 vì: $$\ge k(x_{1}, \dots,  x_{n})$$ có thể đổi thành $$\le (n-k)(\neg x_{1}, \dots, \neg x_{n})$$ và constraint 3 là sự kết hợp của constraint 1 và constraint 2. 





## Mã hóa bằng Sequential Counter

### Giải thích sơ bộ

Để giải bài toán: $$\le k(x_1,\dots, x_n)$$ nghĩa là trong $n$ biến, không được có quá $k$ biến bằng `true`, tác giả xây một sequential counter đếm đi lần lượt từ trái sang phải.


![image.png](images\SequentialCounter\image.png)

#### Hình bên trái
Ở hình bên trái, đếm tuần tự: $x_1 \rightarrow x_2 \rightarrow \dots \rightarrow x_n$. Mỗi trạng thái $s_i$ gồm $k$ bit: $$s_{i,1}, s_{i,2},\dots,s_{i,k}$$ Trong đó: $s_{i,j}=1$ có ý nghĩa là trong $x_1,\dots,x_n$ có ít nhất 1 biến `true`. Ví dụ với $k=3$:$s_i=(1,1,0)$ thì điều này có nghĩa là đã có ít nhất 2 biến `true` nhưng chưa tới 3 nên $s_i = 0$

#### Hình bên phải
Ở hình bên phải, mạch nhận vào:
- biến mới $x_i$
- trạng thái cũ: $s_{i-1,1},s_{i-1,2},\dots,s_{i-1,k}$

Từ đầu vào, mạch tạo ra:
- trạng thái mới: $s_{i,1},s_{i,2},\dots,s_{i,k}$
- cùng bit tràn: $v_i$

Các cổng trong hình có ý nghĩa như sau:
- $\ge 1$ nghĩa là `or`
- `&` nghĩa là `and`

Trong khối này thực hiện 3 công thức sau:
1. $s_{i,1} \iff x_i \lor s_{i-1,i}$:

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $s_{i,1} \rightarrow (x_i \lor s_{i-1,1})$ | **Mệnh đề:** $(x_i \lor s_{i-1,1}) \rightarrow s_{i,1}$ |
| **Ý nghĩa:** Nếu ghi nhận có ít nhất một giá trị `true` tại bước $i$ thì bắt buộc phải có "nguồn gốc": <br> 1. Hoặc biến hiện tại $x_i$ là `true`. <br> 2. Hoặc trước đó ($s_{i-1,1}$) đã có `true`. | **Ý nghĩa:** Nếu thực tế tồn tại một giá trị `true` thì biến trạng thái $s_{i,1}$ bắt buộc phải bật lên để phản ánh đúng thực tế đó: <br> 1. Nếu $x_i = 1$ $\rightarrow$ $s_{i,1} = 1$. <br> 2. Nếu $s_{i-1,1} = 1$ $\rightarrow$ $s_{i,1} = 1$. |
| **Mục đích:** Ngăn chặn việc biến trạng thái tự ý nhận giá trị `true` vô căn cứ (tránh "true nhầm"). | **Mục đích:** Đảm bảo tính lan truyền và kế thừa giá trị logic xuyên suốt chuỗi biến. |


2. $s_{i,j} \iff s_{i-1,j} \lor (x_i \land s_{i-1,j-1}), \quad j>1$: 

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $s_{i,j} \rightarrow s_{i-1,j} \lor (x_i \land s_{i-1,j-1})$ | **Mệnh đề:** $s_{i-1,j} \lor (x_i \land s_{i-1,j-1}) \rightarrow s_{i,j}$ |
| **Ý nghĩa:** Nếu tại bước $i$ ta có ít nhất $j$ biến `true` thì chỉ có hai khả năng đã xảy ra: <br> 1. **Kế thừa:** Ngay từ bước $i-1$ đã có đủ $j$ biến `true` ($s_{i-1,j}$). <br> 2. **Tích lũy mới:** Bước $i-1$ mới có $j-1$ biến `true` và biến hiện tại $x_i$ vừa vặn là biến `true` thứ $j$. | **Ý nghĩa:** Nếu một trong hai điều kiện sau thỏa mãn, trạng thái ít nhất $j$ biến `true` tại bước $i$ phải được xác nhận: <br> 1. Nếu trước đó đã có đủ $j$ biến `true` thì hiển nhiên bước này vẫn có ít nhất $j$ biến. <br> 2. Nếu đã có $j-1$ biến và biến mới $x_i$ cũng là `true` thì tổng cộng đã đạt mốc $j$. |
| **Mục đích:** Đảm bảo biến trạng thái $s_{i,j}$ không tự ý bật lên nếu không thỏa mãn các điều kiện đếm tích lũy. | **Mục đích:** Bắt buộc bộ giải SAT phải cập nhật trạng thái đếm ngay khi điều kiện về số lượng biến `true` được thỏa mãn. |

3. $v_i \iff x_i \land s_{i-1,k}$. 

| Chiều thuận ($\rightarrow$) | Chiều nghịch ($\leftarrow$) |
| :--- | :--- |
| **Mệnh đề:** $v_i \rightarrow x_i \land s_{i-1,k}$ | **Mệnh đề:** $(x_i \land s_{i-1,k}) \rightarrow v_i$ |
| **Ý nghĩa (Điều kiện cần):** <br> Nếu ghi nhận có sự vi phạm ($v_i$) xảy ra tại bước $i$ thì bắt buộc hai điều sau phải cùng xảy ra: <br> 1. Biến hiện tại $x_i$ phải là `true`. <br> 2. Trước đó đã đạt ngưỡng tối đa là $k$ biến `true` ($s_{i-1,k}$). | **Ý nghĩa (Điều kiện đủ):** <br> Nếu thực tế tại bước $i$ ta có $x_i = 1$ và trước đó đã tích lũy đủ $k$ biến `true` thì hệ thống bắt buộc phải kích hoạt biến vi phạm $v_i$. |
| **Mục đích:** Đảm bảo vi phạm chỉ được báo cáo nếu thực sự có biến thứ $k+1$ xuất hiện. | **Mục đích:** Ngăn chặn việc bộ giải SAT lờ đi sự vi phạm; ép giá trị $v_i$ lên `true` để hệ thống nhận diện lỗi. |

